# 🌐 Federated Learning Training — Flower (FedAvg)
**Privacy-Preserving Cancer Detection Project**

This notebook uses the **Flower (`flwr`)** framework to run
Federated Learning across 3 hospital nodes using the `FedAvg` strategy.

It uses the project's `federated_learning/` module with `flwr run`.

---
### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU → Save**
2. Upload these to **MyDrive/FL_Project/** on Google Drive:
   - `ml/` folder (model.py)
   - `federated_learning/` folder (client_app.py, server_app.py, task.py, pyproject.toml)
   - `data/` folder (node1/, node2/, node3/, test/)
3. Run all cells **top to bottom** (Runtime → Run all)

---

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅  Drive mounted.')

## Step 2 — Install Flower

In [ ]:
!pip install -q flwr

import flwr
print(f'✅  Flower version: {flwr.__version__}')

## Step 3 — Check what files exist on Drive

In [ ]:
import os

base = '/content/drive/MyDrive/FL_Project'
print(f'Scanning {base} ...')
print()

for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * 2 * (level + 1)
    for file in files[:5]:
        print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files)-5} more')

## Step 4 — Copy everything from Drive to Colab VM

In [ ]:
import shutil, os

PROJECT_VM = '/content/FL_Project'
DRIVE_PATH = '/content/drive/MyDrive/FL_Project'

os.makedirs(PROJECT_VM, exist_ok=True)

# Folders to copy
for folder in ['ml', 'federated_learning', 'data']:
    src = os.path.join(DRIVE_PATH, folder)
    dst = os.path.join(PROJECT_VM, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  ✅  Copied {folder}/')
    else:
        print(f'  ❌  {folder}/ NOT FOUND on Drive!')

# Create output dirs
os.makedirs(f'{PROJECT_VM}/models', exist_ok=True)
os.makedirs(f'{PROJECT_VM}/results', exist_ok=True)
print()
print('✅  All files copied to Colab VM.')

## Step 5 — Verify all required files

In [ ]:
import os

required_files = [
    'ml/model.py',
    'federated_learning/task.py',
    'federated_learning/client_app.py',
    'federated_learning/server_app.py',
    'federated_learning/pyproject.toml',
    'data/node1/X.npy',
    'data/node1/y.npy',
    'data/node2/X.npy',
    'data/node2/y.npy',
    'data/node3/X.npy',
    'data/node3/y.npy',
    'data/test/X.npy',
    'data/test/y.npy',
]

all_ok = True
for f in required_files:
    path = os.path.join('/content/FL_Project', f)
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ MISSING'
    if not exists:
        all_ok = False
    print(f'  [{status}] {f}')

if not all_ok:
    raise FileNotFoundError('Some files are missing! Upload them to MyDrive/FL_Project/ on Google Drive.')
print('\n✅  All files present. Ready to train!')

## Step 6 — Run Federated Learning Training

This uses `flwr run` to execute the FL pipeline defined in
`federated_learning/` (client_app.py + server_app.py + pyproject.toml).

It runs **10 rounds of FedAvg** across 3 simulated hospital nodes.

In [ ]:
import subprocess, sys, os

os.chdir('/content/FL_Project/federated_learning')

print('=' * 60)
print('  FL Training Starting (10 rounds, FedAvg, Flower)')
print('=' * 60)

process = subprocess.Popen(
    [sys.executable, '-m', 'flwr', 'run', '.', '--stream'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()

if process.returncode == 0:
    print('\n' + '=' * 60)
    print('  ✅  FL Training Complete!')
    print('=' * 60)
else:
    print(f'\n❌  Training failed with return code {process.returncode}')

## Step 7 — Verify model was created

In [ ]:
import os

model_path = '/content/FL_Project/models/federated_model.h5'
metrics_path = '/content/FL_Project/results/fl_metrics.json'

for path in [model_path, metrics_path]:
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f'  ✅  {os.path.basename(path)} ({size:.2f} MB)')
    else:
        print(f'  ❌  {os.path.basename(path)} NOT FOUND')

## Step 8 — Copy model + metrics back to Google Drive

In [ ]:
import shutil, os

items_to_save = {
    '/content/FL_Project/models/federated_model.h5': '/content/drive/MyDrive/FL_Project/models/federated_model.h5',
    '/content/FL_Project/results/fl_metrics.json': '/content/drive/MyDrive/FL_Project/results/fl_metrics.json',
}

for src, dst in items_to_save.items():
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        size = os.path.getsize(src) / (1024 * 1024)
        print(f'  ✅  {os.path.basename(src)} ({size:.2f} MB) → Drive')
    else:
        print(f'  ❌  {src} not found')

print()
print('✅  Done! federated_model.h5 is on your Google Drive.')
print('   Location: /content/drive/MyDrive/FL_Project/models/')
print()
print('Next: download federated_model.h5 from Drive to your local')
print('      models/ folder to complete the project.')

## Step 9 — Display metrics

In [ ]:
import json, os

metrics_path = '/content/FL_Project/results/fl_metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('═' * 55)
    print('  Federated Learning Metrics (Flower FedAvg)')
    print('═' * 55)
    for k, v in metrics.items():
        if isinstance(v, (int, float, str)):
            print(f'  {k:<25}: {v}')
    if 'final_metrics' in metrics:
        print()
        print('  Final Metrics:')
        for k, v in metrics['final_metrics'].items():
            print(f'    {k:<23}: {v}')
    print('═' * 55)
else:
    print('❌  Metrics file not found — check training output above.')